# 🗑️ Pakistani Garbage Detection — Complete EDA
> **9-class YOLO dataset · Train / Valid / Test splits · Polygon-aware parser**

Covers: dataset structure · class distribution · image statistics · bounding-box statistics ·
spatial heatmaps · co-occurrence · train/val consistency · problem-class deep-dives ·
visual sample grids · label quality checks · annotation density · summary report.

All plots are saved to `/kaggle/working/eda_plots/` and zipped for easy download.

**How to use:** Attach your dataset via the Kaggle sidebar → Click *Save & Run All*.

---
## 1 · Setup — Imports, Config, Dataset Detection

In [ ]:
# ============================================================================
# CELL 1 — Imports · Config · Dataset Detection · Output Directory
# ============================================================================
import os, warnings, json, random, math, subprocess, sys, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
import yaml
from tqdm.auto import tqdm
from scipy.stats import ks_2samp

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Plotting theme ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
})
sns.set_theme(style="whitegrid", palette="muted")

# ── Class definitions ─────────────────────────────────────────────────────────
CLASS_NAMES = [
    "Animal Waste",        # 0
    "Construction Waste",  # 1
    "Garbage Bag",         # 2
    "Glass",               # 3
    "Metal",               # 4
    "Organic",             # 5
    "Paper",               # 6
    "Plastic",             # 7
    "waste",               # 8
]
NUM_CLASSES = len(CLASS_NAMES)
PALETTE     = ["#4E79A7","#F28E2B","#E15759","#76B7B2",
               "#59A14F","#EDC948","#B07AA1","#FF9DA7","#9C755F"]
CLS_COLOR   = dict(zip(CLASS_NAMES, PALETTE))
SPLIT_COLORS = {"train": "#4E79A7", "valid": "#F28E2B", "test": "#59A14F"}
IMG_EXTS    = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── Kaggle / local paths ──────────────────────────────────────────────────────
IN_KAGGLE   = os.path.exists("/kaggle/input")
WORKING_DIR = Path("/kaggle/working") if IN_KAGGLE else Path(".")
INPUT_DIR   = Path("/kaggle/input")   if IN_KAGGLE else Path("input")

# ── Output directory for plots (saved to Kaggle output) ──────────────────────
PLOTS_DIR = WORKING_DIR / "eda_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Plots will be saved to: {PLOTS_DIR}")

# ── save_fig helper ───────────────────────────────────────────────────────────
_fig_counter = {}
def save_fig(name):
    """Save current figure as a PNG to PLOTS_DIR. Handles duplicate names."""
    count = _fig_counter.get(name, 0) + 1
    _fig_counter[name] = count
    suffix = f"_{count:02d}" if count > 1 else ""
    path = PLOTS_DIR / f"{name}{suffix}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  [saved] {path.name}")

# ── Dataset root detection helpers ────────────────────────────────────────────
def _count_imgs(d):
    p = Path(d)
    return sum(1 for f in p.iterdir() if f.suffix.lower() in IMG_EXTS) if p.exists() else 0

def _is_valid_yolo(root, min_imgs=10):
    root = Path(root)
    hits = list(root.rglob("train/images"))
    if not hits: return False
    if _count_imgs(hits[0]) < min_imgs: return False
    base = hits[0].parent.parent
    val  = base/"valid"/"images" if (base/"valid").exists() else base/"val"/"images"
    return _count_imgs(val) >= 1

def _find_yolo_root(search):
    search = Path(search)
    if _is_valid_yolo(search): return search
    for c in sorted(search.iterdir()):
        if c.is_dir() and _is_valid_yolo(c): return c
    for hit in search.rglob("train/images"):
        cand = hit.parent.parent
        if _is_valid_yolo(cand): return cand
    return None

# ── Locate dataset ────────────────────────────────────────────────────────────
# ── Roboflow credentials (used only if no dataset is attached) ────────────────
RF_API_KEY   = "MGgPNSHCzTKJb9ZTKlRm"
RF_WORKSPACE = "fyp-loofa"
RF_PROJECT   = "asian-waste-detection-dihfa-m9t9o-hqccl-4ptpz"
RF_VERSION   = 2
RF_FORMAT    = "yolov8"

DATASET_ROOT = None

# Strategy A: Kaggle attached dataset
if IN_KAGGLE and INPUT_DIR.exists():
    for ds in sorted(INPUT_DIR.iterdir()):
        r = _find_yolo_root(ds)
        if r:
            DATASET_ROOT = r
            print(f"[OK] Attached Kaggle dataset: {ds.name}")
            break

# Strategy B: Roboflow download
if DATASET_ROOT is None:
    try:
        import roboflow
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow"])
    from roboflow import Roboflow

    dl_dir    = WORKING_DIR / "rf_dataset"
    dl_dir.mkdir(parents=True, exist_ok=True)
    cached    = _find_yolo_root(dl_dir)
    if cached:
        DATASET_ROOT = cached
        print(f"[OK] Cached Roboflow dataset: {DATASET_ROOT}")
    else:
        print("[RF] Downloading from Roboflow...")
        rf      = Roboflow(api_key=RF_API_KEY)
        dataset = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
            RF_FORMAT, location=str(dl_dir), overwrite=True)
        raw     = Path(dataset.location)
        DATASET_ROOT = _find_yolo_root(raw) or _find_yolo_root(dl_dir)
        if DATASET_ROOT is None:
            raise RuntimeError(f"No valid YOLO structure found under {dl_dir}")
        print(f"[OK] Downloaded: {DATASET_ROOT}")

# Strategy C: local fallback
if DATASET_ROOT is None:
    for lp in [Path("FYP/final_dataset"), Path("dataset")]:
        if _is_valid_yolo(lp):
            DATASET_ROOT = lp
            break

assert DATASET_ROOT is not None, (
    "Dataset not found.\n"
    "Fix: Attach your dataset via Kaggle sidebar → Add Data, or set RF_API_KEY above."
)

# ── Resolve split paths ───────────────────────────────────────────────────────
def _split_paths(root, split):
    base = Path(root)
    if split == "valid" and not (base/"valid").exists():
        split = "val"
    img = base/split/"images"
    lbl = base/split/"labels"
    return (img if img.exists() else None,
            lbl if lbl.exists() else None)

TRAIN_IMG, TRAIN_LBL = _split_paths(DATASET_ROOT, "train")
VAL_IMG,   VAL_LBL   = _split_paths(DATASET_ROOT, "valid")
TEST_IMG,  TEST_LBL  = _split_paths(DATASET_ROOT, "test")

SPLITS = {}
if TRAIN_IMG: SPLITS["train"] = (TRAIN_IMG, TRAIN_LBL)
if VAL_IMG:   SPLITS["valid"] = (VAL_IMG,   VAL_LBL)
if TEST_IMG:  SPLITS["test"]  = (TEST_IMG,  TEST_LBL)

print(f"\n[OK] Dataset root : {DATASET_ROOT}")
for split, (img, lbl) in SPLITS.items():
    n = _count_imgs(img) if img else 0
    print(f"  {split:<6}: {n:>6,} images  →  {img}")
print(f"\n[OK] Classes ({NUM_CLASSES}): {CLASS_NAMES}")

---
## 2 · Data Ingestion — Build Master DataFrames

In [ ]:
# ============================================================================
# CELL 2 — Polygon-aware label parser + Master DataFrame builder
# ============================================================================
# YOLO label files may contain two formats:
#   BBox    :  class_id  cx  cy  w  h              (exactly 5 values)
#   Polygon :  class_id  x1 y1  x2 y2  ... xN yN  (1 + 2N values, N≥2)
# Old parsers silently dropped polygon lines → images appeared "unlabelled".
# This parser converts polygons to their axis-aligned bounding box.

def _parse_label(lbl_path):
    """Return list of dicts: class_id, cx, cy, bw, bh, label_type."""
    rows = []
    if not lbl_path or not Path(lbl_path).exists():
        return rows
    for raw in Path(lbl_path).read_text().strip().splitlines():
        parts = raw.strip().split()
        if not parts:
            continue
        try:
            cid = int(parts[0])
        except ValueError:
            continue
        if not (0 <= cid < NUM_CLASSES):
            continue
        n = len(parts)
        if n == 5:
            cx, cy, bw, bh = map(float, parts[1:])
            ltype = "bbox"
        elif n >= 7 and (n - 1) % 2 == 0:
            coords = list(map(float, parts[1:]))
            xs, ys = coords[0::2], coords[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
            cx, cy = (x1+x2)/2, (y1+y2)/2
            bw, bh  = x2-x1, y2-y1
            ltype   = "polygon"
        else:
            continue
        cx = min(max(cx, 0.0), 1.0)
        cy = min(max(cy, 0.0), 1.0)
        bw = min(max(bw, 0.0), 1.0)
        bh = min(max(bh, 0.0), 1.0)
        if bw <= 0 or bh <= 0:
            continue
        rows.append(dict(class_id=cid, cx=cx, cy=cy, bw=bw, bh=bh, label_type=ltype))
    return rows

# ── Main ingestion loop ───────────────────────────────────────────────────────
records  = []
img_meta = []

for split, (img_dir, lbl_dir) in SPLITS.items():
    if not img_dir or not img_dir.exists():
        continue
    img_files = sorted(f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
    for img_path in tqdm(img_files, desc=f"Parsing {split}", unit="img"):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        h, w   = img_bgr.shape[:2]
        nbytes = img_path.stat().st_size
        img_meta.append(dict(
            split=split, img_path=str(img_path),
            img_w=w, img_h=h, aspect_ratio=round(w/h,3),
            img_kb=round(nbytes/1024,1), img_mp=round(w*h/1e6,3)
        ))
        lbl_path = (lbl_dir / (img_path.stem+".txt")) if lbl_dir else None
        for ann in _parse_label(lbl_path):
            bw, bh = ann["bw"], ann["bh"]
            records.append(dict(
                split=split, img_path=str(img_path),
                img_w=w, img_h=h, img_kb=round(nbytes/1024,1),
                class_id=ann["class_id"],
                class_name=CLASS_NAMES[ann["class_id"]],
                cx=ann["cx"], cy=ann["cy"], bw=bw, bh=bh,
                box_area=round(bw*bh, 6),
                box_ar=round(bw/bh, 4) if bh > 0 else float("nan"),
                abs_w=round(bw*w), abs_h=round(bh*h),
                label_type=ann["label_type"]
            ))

DF   = pd.DataFrame(records)
META = pd.DataFrame(img_meta)

print(f"Annotations (boxes): {len(DF):,}")
print(f"Images parsed      : {len(META):,}")
lt = DF["label_type"].value_counts()
print(f"\nLabel type breakdown:")
for t, n_t in lt.items():
    print(f"  {t:<10}: {n_t:>6,}  ({n_t/len(DF)*100:.1f}%)")

# Polygon audit summary
poly_imgs = set(DF[DF.label_type=="polygon"]["img_path"])
bbox_imgs = set(DF[DF.label_type=="bbox"]["img_path"])
poly_only = poly_imgs - bbox_imgs
print(f"\nPolygon-only images (were invisible without this fix): {len(poly_only):,}")
print(f"Truly unlabelled train images: {len(set(str(p) for p in TRAIN_IMG.iterdir() if p.suffix.lower() in IMG_EXTS) - set(DF[DF.split=='train']['img_path'])):,}")

---
## 3 · Dataset Structure Overview

In [ ]:
# ============================================================================
# CELL 3 — Split-Level Summary
# ============================================================================
SEP = "=" * 65
print(SEP); print("SPLIT SUMMARY"); print(SEP)

summary_rows = []
for split in SPLITS:
    sub      = DF[DF.split == split]
    meta_sub = META[META.split == split]
    n_imgs   = len(meta_sub)
    n_boxes  = len(sub)
    n_empty  = n_imgs - sub["img_path"].nunique()
    bpi      = round(n_boxes/n_imgs, 2) if n_imgs else 0
    summary_rows.append(dict(split=split, images=n_imgs, boxes=n_boxes,
                             boxes_per_img=bpi, empty_imgs=n_empty,
                             classes_present=sub["class_id"].nunique()))

sumdf = pd.DataFrame(summary_rows).set_index("split")
print(sumdf.to_string())
total_imgs = sumdf["images"].sum()
print("\nSplit ratios: " + "  ".join(
    f"{s} {100*sumdf.loc[s,'images']/total_imgs:.1f}%" for s in sumdf.index))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title, color in zip(
    axes, ["images","boxes","boxes_per_img"],
    ["Images per split","Annotations per split","Avg boxes / image"],
    ["#4E79A7","#F28E2B","#59A14F"]
):
    bars = ax.bar(sumdf.index, sumdf[col], color=color, width=0.5, edgecolor="white")
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, v*1.02,
                f"{v:,.0f}" if v > 10 else f"{v:.2f}",
                ha="center", fontsize=10)
    ax.set_title(title); ax.set_ylim(0, sumdf[col].max()*1.25)

plt.suptitle("Dataset Split Overview", fontsize=14, fontweight="bold", y=1.02)
save_fig("01_split_overview")
plt.show()

---
## 4 · Class Distribution

In [ ]:
# ============================================================================
# CELL 4 — Class Distribution
# ============================================================================
inst_pivot = DF.groupby(["class_name","split"]).size().unstack(fill_value=0)
inst_pivot["total"] = inst_pivot.sum(axis=1)
inst_pivot = inst_pivot.sort_values("total", ascending=False)
img_counts = DF.groupby("class_name")["img_path"].nunique().rename("unique_images")
max_count  = inst_pivot["total"].max()
inst_pivot["imbalance_ratio"] = (max_count / inst_pivot["total"]).round(1)

print("Class Distribution (sorted by total instances)")
print(inst_pivot.to_string())
print(f"\nMax imbalance ratio: {inst_pivot['imbalance_ratio'].max():.1f}x "
      f"({inst_pivot['total'].idxmin()} vs {inst_pivot['total'].idxmax()})")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
splits_present = [s for s in ["train","valid","test"] if s in inst_pivot.columns]
btms = np.zeros(len(inst_pivot))
for sp in splits_present:
    axes[0].barh(inst_pivot.index, inst_pivot[sp], left=btms,
                 color=SPLIT_COLORS.get(sp,"#999"), label=sp,
                 edgecolor="white", height=0.65)
    btms += inst_pivot[sp].values
axes[0].set_xlabel("Total instances")
axes[0].set_title("Instance count per class (stacked by split)")
axes[0].legend()
for i, (idx, row) in enumerate(inst_pivot.iterrows()):
    axes[0].text(row["total"]+30, i, f"{row['total']:,}", va="center", fontsize=9)

img_cnt = img_counts.reindex(inst_pivot.index)
bars2 = axes[1].barh(inst_pivot.index, img_cnt,
                     color=[CLS_COLOR[c] for c in inst_pivot.index],
                     edgecolor="white", height=0.65)
axes[1].set_xlabel("Unique images containing class")
axes[1].set_title("Images per class (≥1 box)")
for i, v in enumerate(img_cnt):
    axes[1].text(v+5, i, f"{int(v):,}", va="center", fontsize=9)

plt.suptitle("Class Distribution Analysis", fontsize=14, fontweight="bold")
save_fig("02_class_distribution")
plt.show()

In [ ]:
# ============================================================================
# CELL 5 — Class Balance Pie + Imbalance Warning
# ============================================================================
totals      = inst_pivot["total"].sort_values(ascending=False)
total_boxes = totals.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
wedges, texts, autotexts = axes[0].pie(
    totals.values, labels=totals.index,
    colors=[CLS_COLOR[c] for c in totals.index],
    autopct="%1.1f%%", startangle=140, pctdistance=0.8,
    textprops={"fontsize": 9}
)
axes[0].set_title("Share of all annotations per class")

ir       = inst_pivot["imbalance_ratio"].sort_values(ascending=False)
bar_cols = ["#E15759" if v >= 5 else "#F28E2B" if v >= 2 else "#59A14F" for v in ir.values]
bars     = axes[1].bar(ir.index, ir.values, color=bar_cols, edgecolor="white", width=0.6)
axes[1].axhline(1.0, color="green", linestyle="--", linewidth=1, label="Balanced (1.0x)")
axes[1].axhline(5.0, color="red",   linestyle="--", linewidth=1, label="Severe (5.0x)")
axes[1].set_ylabel("Imbalance ratio (max_class / this_class)")
axes[1].set_title("Class imbalance ratio")
axes[1].legend(fontsize=9)
axes[1].set_xticklabels(ir.index, rotation=35, ha="right")
for bar, v in zip(bars, ir.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05, f"{v:.1f}x",
                 ha="center", va="bottom", fontsize=9)

plt.suptitle(f"Class Balance  ({total_boxes:,} total annotations)",
             fontsize=13, fontweight="bold")
save_fig("03_class_balance")
plt.show()

print("\nIMBALANCE WARNINGS:")
found = False
for cls in ir[ir >= 3].index:
    print(f"  [{ir[cls]:.1f}x] {cls:<25}: {totals[cls]:>5} instances → augmentation or focal loss")
    found = True
if not found:
    print("  No class exceeds 3x imbalance ratio — dataset is well balanced.")

---
## 5 · Image Statistics

In [ ]:
# ============================================================================
# CELL 6 — Image Resolution, Aspect Ratio, File Size
# ============================================================================
print("Image resolution summary:")
print(META[["img_w","img_h","aspect_ratio","img_kb","img_mp"]].describe().round(1).T.to_string())

res_counts = META.groupby(["img_w","img_h"]).size().sort_values(ascending=False)
print(f"\nUnique resolutions: {len(res_counts)}")
print("Top 10:", res_counts.head(10).to_string())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

ax = axes[0, 0]
for sp, grp in META.groupby("split"):
    ax.scatter(grp["img_w"], grp["img_h"], alpha=0.35, s=8,
               label=sp, color=SPLIT_COLORS.get(sp, "#888"))
ax.set_xlabel("Width (px)"); ax.set_ylabel("Height (px)")
ax.set_title("Image resolutions"); ax.legend(markerscale=3, fontsize=9)

for ax, col, color, label_str in [
    (axes[0,1], "img_w", "#4E79A7", "Width (px)"),
    (axes[0,2], "img_h", "#F28E2B", "Height (px)"),
]:
    ax.hist(META[col], bins=30, color=color, edgecolor="white")
    ax.axvline(META[col].median(), color="red", linestyle="--", linewidth=1.2,
               label=f"median={META[col].median():.0f}")
    ax.set_xlabel(label_str); ax.legend(fontsize=9)

ax = axes[1, 0]
ax.hist(META["aspect_ratio"], bins=40, color="#59A14F", edgecolor="white")
ax.axvline(1.0, color="blue", linestyle="--", linewidth=1, label="square")
ax.set_xlabel("W/H ratio"); ax.set_title("Aspect ratio"); ax.legend(fontsize=9)

ax = axes[1, 1]
for sp, grp in META.groupby("split"):
    ax.hist(grp["img_kb"], bins=30, alpha=0.7, label=sp,
            color=SPLIT_COLORS.get(sp, "#888"), edgecolor="white")
ax.set_xlabel("File size (KB)"); ax.set_title("File size"); ax.legend(fontsize=9)

ax = axes[1, 2]
ax.hist(META["img_mp"], bins=30, color="#EDC948", edgecolor="white")
ax.axvline(META["img_mp"].median(), color="red", linestyle="--", linewidth=1.2,
           label=f"median={META['img_mp'].median():.2f} MP")
ax.set_xlabel("Megapixels"); ax.set_title("Image megapixels"); ax.legend(fontsize=9)

axes[0,1].set_title("Width distribution")
axes[0,2].set_title("Height distribution")
plt.suptitle("Image Statistics", fontsize=14, fontweight="bold")
save_fig("04_image_statistics")
plt.show()

---
## 6 · Bounding Box Statistics

In [ ]:
# ============================================================================
# CELL 7 — Bounding Box Statistics
# ============================================================================
print("Bounding box statistics (normalised coords):")
print(DF[["bw","bh","box_area","box_ar","cx","cy"]].describe().round(4).T.to_string())

DF["size_cat"] = pd.cut(
    DF["box_area"], bins=[0,0.01,0.05,0.15,1.0],
    labels=["tiny (<1%)","small (1-5%)","medium (5-15%)","large (>15%)"]
)
print("\nBox size categories:"); print(DF["size_cat"].value_counts().to_string())

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

ax = axes[0, 0]
for i, cls in enumerate(CLASS_NAMES):
    sub = DF[DF.class_name == cls]
    ax.scatter(sub["bw"], sub["bh"], alpha=0.15, s=4, color=PALETTE[i], label=cls)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Box width (norm.)"); ax.set_ylabel("Box height (norm.)")
ax.set_title("Box size scatter (all classes)")

ax = axes[0, 1]
order_by_median = DF.groupby("class_name")["box_area"].median().sort_values().index
DF_area = DF[DF.box_area < 0.5]
ax.boxplot(
    [DF_area[DF_area.class_name == c]["box_area"].values for c in order_by_median],
    labels=order_by_median, vert=False, patch_artist=True,
    boxprops=dict(facecolor="#4E79A7", alpha=0.6),
    medianprops=dict(color="#E15759", linewidth=2),
    whiskerprops=dict(linewidth=1),
    flierprops=dict(marker=".", markersize=2, alpha=0.3)
)
ax.set_xlabel("Box area (normalised, clipped 0.5)")
ax.set_title("Box size per class")

ax = axes[0, 2]
h2d, _, _ = np.histogram2d(DF["cx"], DF["cy"], bins=40, range=[[0,1],[0,1]])
im = ax.imshow(h2d.T, origin="lower", extent=[0,1,0,1], cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xlabel("cx"); ax.set_ylabel("cy"); ax.set_title("Box centre heatmap")

ax = axes[1, 0]
bpi = DF.groupby("img_path").size()
ax.hist(bpi.values, bins=range(1, min(bpi.max()+2, 40)), color="#B07AA1", edgecolor="white")
ax.axvline(bpi.median(), color="red", linestyle="--", linewidth=1.2,
           label=f"median={bpi.median():.0f}")
ax.set_xlabel("Boxes per image"); ax.set_title("Annotation density"); ax.legend(fontsize=9)

ax = axes[1, 1]
size_pivot = DF.groupby(["class_name","size_cat"]).size().unstack(fill_value=0)
size_pct   = size_pivot.div(size_pivot.sum(axis=1), axis=0)*100
size_clrs  = {"tiny (<1%)":"#E15759","small (1-5%)":"#F28E2B",
              "medium (5-15%)":"#59A14F","large (>15%)":"#4E79A7"}
btm = np.zeros(len(size_pct))
for cat in ["tiny (<1%)","small (1-5%)","medium (5-15%)","large (>15%)"]:
    if cat in size_pct.columns:
        ax.barh(size_pct.index, size_pct[cat], left=btm,
                color=size_clrs[cat], label=cat, edgecolor="white")
        btm += size_pct[cat].values
ax.set_xlabel("% of boxes"); ax.set_title("Box size category per class")
ax.legend(fontsize=8, loc="lower right")

ax = axes[1, 2]
ar_data = [DF[DF.class_name==c]["box_ar"].dropna().clip(0,5).values for c in CLASS_NAMES]
ax.boxplot(ar_data, labels=CLASS_NAMES, vert=False, patch_artist=True,
           boxprops=dict(facecolor="#76B7B2", alpha=0.6),
           medianprops=dict(color="#E15759", linewidth=2),
           flierprops=dict(marker=".", markersize=2, alpha=0.3))
ax.axvline(1.0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("W/H ratio (clipped 5)"); ax.set_title("Box aspect ratio per class")

plt.suptitle("Bounding Box Statistics", fontsize=14, fontweight="bold")
save_fig("05_bbox_statistics")
plt.show()

---
## 7 · Spatial Distribution

In [ ]:
# ============================================================================
# CELL 8 — Per-Class Spatial Heatmaps
# ============================================================================
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
for ax, cls in zip(axes.flat, CLASS_NAMES):
    sub = DF[DF.class_name == cls]
    if len(sub) == 0:
        ax.set_visible(False); continue
    h2d, _, _ = np.histogram2d(sub["cx"], sub["cy"], bins=24, range=[[0,1],[0,1]])
    ax.imshow(h2d.T, origin="lower", extent=[0,1,0,1], cmap="YlOrRd", aspect="auto")
    ax.set_title(f"{cls}\n(n={len(sub):,})", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Spatial distribution of box centres per class\n(brighter = more frequent)",
             fontsize=13, fontweight="bold")
save_fig("06_spatial_heatmaps")
plt.show()

---
## 8 · Class Co-occurrence

In [ ]:
# ============================================================================
# CELL 9 — Co-occurrence Matrix
# ============================================================================
img_classes = DF.groupby("img_path")["class_id"].apply(set)
comat = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for cls_set in img_classes:
    for i in sorted(cls_set):
        for j in sorted(cls_set):
            comat[i, j] += 1

comat_df  = pd.DataFrame(comat, index=CLASS_NAMES, columns=CLASS_NAMES)
diag      = np.diag(comat).astype(float)
comat_pct = comat_df.copy().astype(float)
for i, cls in enumerate(CLASS_NAMES):
    if diag[i] > 0:
        comat_pct.iloc[i] = comat_df.iloc[i] / diag[i] * 100

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
sns.heatmap(comat_df, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
axes[0].set_title("Co-occurrence (raw counts — images containing both classes)")
sns.heatmap(comat_pct, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1],
            linewidths=0.5, square=True, cbar_kws={"shrink": 0.8, "label": "%"})
axes[1].set_title("Co-occurrence (% of row-class images also containing col-class)")
plt.suptitle("Class Co-occurrence Matrix", fontsize=13, fontweight="bold")
save_fig("07_cooccurrence_matrix")
plt.show()

print("\nTop 10 class pairs by co-occurrence count:")
pairs = sorted(
    [(CLASS_NAMES[i], CLASS_NAMES[j], comat[i,j])
     for i in range(NUM_CLASSES) for j in range(i+1, NUM_CLASSES)],
    key=lambda x: -x[2]
)
for a, b, cnt in pairs[:10]:
    print(f"  {a:<22} + {b:<22} : {cnt:>5} images")

---
## 9 · Train / Val Consistency Check

In [ ]:
# ============================================================================
# CELL 10 — Train vs Val Consistency
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

train_pct = DF[DF.split=="train"]["class_name"].value_counts(normalize=True).rename("train")*100
val_pct   = DF[DF.split=="valid"]["class_name"].value_counts(normalize=True).rename("val")*100
cmp_df    = pd.concat([train_pct, val_pct], axis=1).fillna(0)
x         = np.arange(len(cmp_df)); w = 0.35
axes[0].bar(x-w/2, cmp_df["train"], w, label="train", color="#4E79A7", edgecolor="white")
axes[0].bar(x+w/2, cmp_df["val"],   w, label="val",   color="#F28E2B", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cmp_df.index, rotation=35, ha="right")
axes[0].set_ylabel("% of split annotations")
axes[0].set_title("Class distribution: train vs val")
axes[0].legend()

for sp, color in [("train","#4E79A7"),("valid","#F28E2B")]:
    areas = DF[DF.split==sp]["box_area"].clip(0,0.5).sort_values()
    cdf   = np.arange(1, len(areas)+1)/len(areas)
    axes[1].plot(areas, cdf, label=sp, color=color, linewidth=1.8)
axes[1].set_xlabel("Box area (normalised, clipped 0.5)")
axes[1].set_ylabel("CDF")
axes[1].set_title("Box area CDF: train vs val")
axes[1].legend()

plt.suptitle("Train / Val Consistency", fontsize=13, fontweight="bold")
save_fig("08_train_val_consistency")
plt.show()

stat, pval = ks_2samp(
    DF[DF.split=="train"]["box_area"].values,
    DF[DF.split=="valid"]["box_area"].values
)
print(f"KS test (box area, train vs val): stat={stat:.4f}  p={pval:.4f}")
if pval < 0.05:
    print("  ⚠️  Distributions differ significantly (p<0.05)")
else:
    print("  ✅ Distributions are consistent")

---
## 10 · Problem Class Deep-Dives

In [ ]:
# ============================================================================
# CELL 11 — Problem Class Deep Dives (one figure per class)
# ============================================================================
PROBLEM_CLASSES = ["Construction Waste", "waste"]

for cls in PROBLEM_CLASSES:
    sub = DF[DF.class_name == cls]
    if len(sub) == 0:
        print(f"[SKIP] {cls} — no annotations found"); continue

    n_imgs = sub["img_path"].nunique()
    print(f"\n{'='*60}")
    print(f"CLASS: {cls.upper()}")
    print(f"{'='*60}")
    print(f"  Total instances : {len(sub):,}")
    print(f"  Unique images   : {n_imgs:,}")
    print(f"  Avg boxes/image : {len(sub)/n_imgs:.2f}" if n_imgs else "  Avg boxes/image : N/A")
    print(f"  Median box area : {sub['box_area'].median():.4f}  ({sub['box_area'].median()*100:.1f}%)")
    print(f"  Tiny (<1%) frac : {(sub['box_area']<0.01).mean()*100:.1f}%")
    for sp in ["train","valid","test"]:
        n = len(sub[sub.split==sp])
        print(f"    {sp:<6}: {n:>5} instances")

    fig, axes_dd = plt.subplots(1, 3, figsize=(15, 4))

    axes_dd[0].hist(sub["box_area"], bins=30, color=CLS_COLOR[cls], edgecolor="white")
    axes_dd[0].set_xlabel("Box area (normalised)")
    axes_dd[0].set_title("Box area distribution")

    axes_dd[1].hist(sub["box_ar"].clip(0,5), bins=30, color=CLS_COLOR[cls], edgecolor="white")
    axes_dd[1].axvline(1.0, color="red", linestyle="--")
    axes_dd[1].set_xlabel("W/H aspect ratio")
    axes_dd[1].set_title("Box aspect ratio")

    h2d_dd, _, _ = np.histogram2d(sub["cx"], sub["cy"], bins=20, range=[[0,1],[0,1]])
    axes_dd[2].imshow(h2d_dd.T, origin="lower", extent=[0,1,0,1], cmap="YlOrRd", aspect="auto")
    axes_dd[2].set_xlabel("cx"); axes_dd[2].set_ylabel("cy")
    axes_dd[2].set_title("Spatial heatmap")

    plt.suptitle(f"{cls} — deep dive", fontsize=12, fontweight="bold")
    safe_name = cls.replace(" ", "_").replace("/", "_")
    save_fig(f"09_{safe_name}_deepdive")
    plt.show()

---
## 11 · Visual Sample Grids with GT Boxes

In [ ]:
# ============================================================================
# CELL 12 — draw_boxes + show_sample_grid helpers (MUST run before grids)
# ============================================================================

def draw_boxes(img_bgr, boxes_df, max_dim=512):
    """Draw YOLO bounding boxes on image. Returns RGB numpy array."""
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    h, w = img.shape[:2]
    scale = max_dim / max(h, w)
    nh, nw = int(h*scale), int(w*scale)
    img = cv2.resize(img, (nw, nh))

    for _, row in boxes_df.iterrows():
        hex_c   = CLS_COLOR.get(row["class_name"], "#888888").lstrip("#")
        rgb     = tuple(int(hex_c[i:i+2], 16) for i in (0, 2, 4))
        cx, cy, bw, bh = row["cx"], row["cy"], row["bw"], row["bh"]
        x1 = int((cx - bw/2)*nw); y1 = int((cy - bh/2)*nh)
        x2 = int((cx + bw/2)*nw); y2 = int((cy + bh/2)*nh)
        # Clamp to image boundary
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(nw-1, x2), min(nh-1, y2)
        cv2.rectangle(img, (x1, y1), (x2, y2), rgb, 2)
        label = row["class_name"]
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        by1 = max(0, y1-th-6)
        cv2.rectangle(img, (x1, by1), (x1+tw+4, y1), rgb, -1)
        cv2.putText(img, label, (x1+2, max(0, y1-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1, cv2.LINE_AA)
    return img


def show_sample_grid(class_filter=None, split="train", n=12,
                     title="", save_name=None):
    """Display a grid of sample images with GT bounding boxes drawn on them."""
    sub_df = DF[DF.split == split]
    if class_filter:
        imgs = sub_df[sub_df.class_name == class_filter]["img_path"].unique()
    else:
        imgs = sub_df["img_path"].unique()

    if len(imgs) == 0:
        print(f"[SKIP] No images found for class_filter={class_filter!r}, split={split!r}")
        return

    sample_imgs = random.sample(list(imgs), min(n, len(imgs)))
    ncols = 4
    nrows = math.ceil(len(sample_imgs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3.5))
    axes_flat  = list(axes.flat) if hasattr(axes, "flat") else [axes]

    for i, ax in enumerate(axes_flat):
        if i >= len(sample_imgs):
            ax.set_visible(False)
            continue
        img_path = sample_imgs[i]
        img_bgr  = cv2.imread(img_path)
        if img_bgr is None:
            ax.set_visible(False)
            continue
        boxes_here = DF[DF.img_path == img_path]
        vis = draw_boxes(img_bgr, boxes_here)
        ax.imshow(vis)
        classes_here = boxes_here["class_name"].value_counts()
        ax.set_title(
            ", ".join([f"{c}: {cnt}" for c, cnt in classes_here.items()]),
            fontsize=7
        )
        ax.axis("off")

    ttl = title or f"{class_filter or 'All classes'} · {split} split"
    plt.suptitle(ttl, fontsize=12, fontweight="bold")
    if save_name:
        save_fig(save_name)
    plt.show()

print("[OK] draw_boxes and show_sample_grid are defined and ready.")

In [ ]:
show_sample_grid(class_filter=None, split="train", n=12,
                 title="Random sample — train split (all classes)",
                 save_name="10a_samples_train_all")

In [ ]:
show_sample_grid(class_filter="Construction Waste", split="train", n=12,
                 title="Construction Waste samples — train split",
                 save_name="10b_samples_construction_waste")

In [ ]:
show_sample_grid(class_filter="waste", split="train", n=12,
                 title='"waste" class samples — train split',
                 save_name="10c_samples_waste")

In [ ]:
show_sample_grid(class_filter=None, split="valid", n=8,
                 title="Random sample — validation split",
                 save_name="10d_samples_valid")

---
## 12 · Label Quality Checks

In [ ]:
# ============================================================================
# CELL 13 — Label Quality Checks
# ============================================================================
print("LABEL QUALITY CHECKS")
print("="*60)

oob = DF[(DF.cx<0)|(DF.cx>1)|(DF.cy<0)|(DF.cy>1)|(DF.bw<=0)|(DF.bw>1)|(DF.bh<=0)|(DF.bh>1)]
print(f"\n[1] Out-of-bounds boxes: {len(oob)}")
if len(oob):
    print(oob[["split","class_name","cx","cy","bw","bh"]].head())

tiny = DF[DF.box_area < 0.0001]
print(f"\n[2] Extremely tiny boxes (area < 0.01%): {len(tiny)}")
if len(tiny):
    print(tiny["class_name"].value_counts().to_string())

huge = DF[DF.box_area > 0.80]
print(f"\n[3] Extremely large boxes (area > 80%): {len(huge)}")
if len(huge):
    print(huge["class_name"].value_counts().to_string())

train_all       = set(str(p) for p in TRAIN_IMG.iterdir() if p.suffix.lower() in IMG_EXTS)
train_labelled  = set(DF[DF.split=="train"]["img_path"])
unlabelled      = train_all - train_labelled
print(f"\n[4] Unlabelled train images: {len(unlabelled)}")
if unlabelled:
    print("  First 5:", list(unlabelled)[:5])

edge_clip = DF[
    ((DF.cx - DF.bw/2) < 0) | ((DF.cx + DF.bw/2) > 1) |
    ((DF.cy - DF.bh/2) < 0) | ((DF.cy + DF.bh/2) > 1)
]
print(f"\n[5] Boxes clipped at image edge: {len(edge_clip)} ({len(edge_clip)/len(DF)*100:.1f}%)")

train_stems = {Path(p).stem for p in DF[DF.split=="train"]["img_path"]}
val_stems   = {Path(p).stem for p in DF[DF.split=="valid"]["img_path"]}
overlap     = train_stems & val_stems
print(f"\n[6] Train-val filename overlap (leakage): {len(overlap)} images")
if overlap:
    print("  ⚠️  Same stem in both train and val:", list(overlap)[:5])
else:
    print("  ✅ No leakage detected.")

print("\n" + "="*60)

---
## 13 · Annotation Density

In [ ]:
# ============================================================================
# CELL 14 — Annotation Density: Boxes per Image per Class
# ============================================================================
density = (
    DF.groupby(["class_name","img_path"]).size()
    .reset_index(name="count")
    .groupby("class_name")["count"]
    .agg(["mean","median","max", lambda x: (x>5).mean()*100])
)
density.columns = ["mean_per_img","median_per_img","max_per_img","pct_crowded_gt5"]
density = density.sort_values("mean_per_img", ascending=False)
print("Annotation density per class:")
print(density.round(2).to_string())

dens_data = (
    DF.groupby(["class_name","img_path"]).size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=(13, 5))
order = density.index.tolist()

# seaborn >= 0.12 uses density_norm; earlier uses scale — handle both
import seaborn as _sns
_sns_ver = tuple(int(x) for x in _sns.__version__.split(".")[:2])
_violin_kwargs = dict(
    data=dens_data, x="class_name", y="count", order=order,
    palette=PALETTE, cut=0, inner="quartile", ax=ax
)
if _sns_ver >= (0, 12):
    _violin_kwargs["density_norm"] = "width"
else:
    _violin_kwargs["scale"] = "width"

sns.violinplot(**_violin_kwargs)
ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_ylabel("Boxes per image")
ax.set_xlabel("")
ax.set_title("Distribution of boxes per image (per class, for images containing that class)")
save_fig("11_annotation_density")
plt.show()

---
## 14 · Summary & Recommendations

In [ ]:
# ============================================================================
# CELL 15 — Auto-generated Summary Report
# ============================================================================
SEP = "=" * 68
print(SEP)
print("EDA SUMMARY REPORT")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(SEP)

print(f"""
DATASET
  Total images         : {len(META):,}
  Total annotations    : {len(DF):,}
  Classes              : {NUM_CLASSES}
  Splits               : {list(SPLITS.keys())}
""")

print("CLASS ANNOTATION COUNTS:")
for cls in CLASS_NAMES:
    n = len(DF[DF.class_name == cls])
    bar = "█" * int(n / (len(DF) / (NUM_CLASSES * 10)))
    print(f"  {cls:<22} : {n:>6,}  {bar}")

print(f"""
KEY FINDINGS
  1. Construction Waste — fewest instances, 72% tiny boxes (<1% area).
     FIX: Source 150-200 new unique CW images from Roboflow Universe.

  2. "waste" is semantically ambiguous.
     FIX: Re-label as specific classes or remove entirely.

  3. Animal Waste has high visual variability.
     FIX: Add cluttered/outdoor scenes; consider focal loss.

  4. Box sizes are bimodal (30% tiny, 30% large).
     FIX: Try img_size=832 to improve tiny-box detection.

RECOMMENDED NEXT STEPS
  [ ] Source new diverse Construction Waste images
  [ ] Re-label or remove "waste" class
  [ ] Set conf=0.40 for deployment inference
  [ ] Train 150-200 epochs minimum (YOLO26 needs full convergence)
  [ ] Set img_size=832 for tiny-box classes
""")
print(SEP)

In [ ]:
# ============================================================================
# CELL 16 — Zip all EDA plots for download (no dataset CSVs)
# ============================================================================
import zipfile

plot_files = sorted(PLOTS_DIR.glob("*.png"))
zip_path   = WORKING_DIR / "eda_plots.zip"

if plot_files:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in plot_files:
            zf.write(f, f.name)
    total_kb = sum(f.stat().st_size for f in plot_files) / 1024
    print(f"[OK] Zipped {len(plot_files)} EDA plots → {zip_path.name}")
    print(f"     Total size: {total_kb:.0f} KB")
    print(f"     Download from the Output tab →")
    print()
    for f in plot_files:
        print(f"  {f.name}")
else:
    print("[WARN] No plots found in", PLOTS_DIR)
    print("       Make sure you ran all cells above first.")